# Building a Character-Level Language Model with LSTMs

## What is this Notebook?
In this notebook, we are shifting our focus from predicting continuous numerical values (regression) to understanding and generating human language. We will build a **Character-Level Language Model** using a Long Short-Term Memory (LSTM) network. 

Instead of treating text as words, this model operates on the most fundamental level: individual characters (letters, punctuation marks, and spaces). By observing a sequence of characters, the network learns to predict which specific character is most likely to appear next.

## The General Working Principle
At its core, a language model is a probability engine. The architecture follows a specific pipeline to process discrete text:

1. **Tokenization (Character-to-Integer):** Neural networks cannot read letters. We first identify every unique character in our dataset (the vocabulary) and assign each a unique integer ID.
2. **The Embedding Layer:** Unlike raw numbers, these integer IDs represent categories. The Embedding layer transforms these IDs into dense mathematical vectors, allowing the network to learn relationships between different characters.
3. **The LSTM Core:** The sequence of vectors is fed into the LSTM. The network uses its hidden state to remember the historical context (e.g., "I just saw the letters 't' and 'h', so the next letter is probably 'e' or 'a'").
4. **Classification:** The output of the LSTM is passed through a Linear layer, which scores every possible character in our vocabulary. We use these scores to calculate the probability of the next character.

## The Data
We are using raw, unstructured text data-specifically, the "Tiny Shakespeare" dataset, which contains a collection of the playwright's works. 

Unlike standard datasets that come with clear features and labels, we have to create our own. We use a **Sliding Window** approach: we take a chunk of text (e.g., 100 characters) as our input ($X$), and the exact same chunk shifted one character to the right as our target ($Y$). This transforms a massive wall of text into a supervised learning problem.

## The Goal
Our objective is to train the LSTM to minimize the **Cross-Entropy Loss**. Once the loss is low enough, the model will have internalized the fundamental rules of English spelling, vocabulary, and basic grammar. We can then use the trained model for **Inference (Text Generation)**: feeding it a starting seed (a "prime" string) and asking it to write a completely new, original text character by character.

## Intuition: How does it learn?
Think of this model as an incredibly advanced smartphone auto-complete. When initialized, the model doesn't know what a word is, what a space is, or even what a vowel is. 

Through training, the LSTM's memory cells learn complex patterns. In the short term, it learns basic spelling (if it sees `q`, it expects a `u`). In the medium term, it learns words (if it sees `k n i g h`, it expects a `t`). In the long term, it learns formatting—for example, if it opens a parenthesis, its memory cell will hold onto that context for dozens of steps until it generates a closing parenthesis.

In [50]:
import torch
from torch import nn
import numpy as np
import requests
import torch_directml
import torch.nn.functional as F

## Understanding the Hyperparameters

Before we process our data or build our model, we must define the hyperparameters that will dictate how our network learns and operates. 

* **`SEQ_LENGTH = 100`**: The context window. This defines how many characters the LSTM will look at simultaneously to predict the next character. A longer sequence allows the model to capture longer dependencies (like multi-syllable words or punctuation pairing), but takes more memory to train.
* **`BATCH_SIZE = 64`**: The number of sequences processed in parallel during one training step. 
* **`EMBEDDING_DIM = 128`**: The size of the mathematical vector used to represent each character. Instead of just a single integer, each character becomes a 128-dimensional point in space, allowing the network to learn similarities between different characters (e.g., grouping vowels together).
* **`HIDDEN_SIZE = 256`**: The memory capacity of our LSTM cells. A larger hidden size allows the network to remember more complex, nuanced patterns in the text, but increases computational cost and the risk of overfitting.
* **`NUM_LAYERS = 2`**: The depth of the network. Stacking two LSTMs on top of each other allows the model to learn hierarchical features (e.g., Layer 1 learns spelling, Layer 2 learns grammar).
* **`LEARNING_RATE = 1e-3`**: Controls how aggressively the optimizer updates the network's weights during training.
* **`NUM_EPOCHS = 20`**: The number of complete passes the training loop will make over the entire dataset.
* **`DEVICE = "cpu"`**: Defines the hardware accelerator used for matrix calculations. (Can be changed to `"cuda"` if a GPU is available).
* **`CLIP = 5`**: The threshold for *Gradient Clipping*. As mentioned in the introduction, LSTMs can suffer from exploding gradients. This parameter explicitly caps the maximum size of a gradient update to keep training stable.
* **`URL`**: The source link to our raw training data (in this case, the Tiny Shakespeare dataset).

In [51]:
SEQ_LENGTH = 100
BATCH_SIZE = 64
EMBEDDING_DIM = 128
HIDDEN_SIZE = 256
LEARNING_RATE = 1e-3
NUM_EPOCHS = 20
NUM_LAYERS = 2
DEVICE = "cpu"
CLIP = 5

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

## Data Loading and Tokenization

Unlike regression tasks where data is already numerical, language models require an extra preprocessing step to translate text into numbers. This process is called **Tokenization**. In this notebook, we are performing character-level tokenization.

Here is what the code achieves step-by-step:

* **Fetching and Truncating:** We download the raw text data from the provided URL. For the sake of faster training and prototyping in this notebook, we slice the text to use only the first 100,000 characters.
* **Building the Vocabulary:** By applying `set(text)`, we extract every unique character present in our dataset (including spaces, line breaks, and punctuation). The total number of these unique characters becomes our `VOCAB_SIZE`.
* **Mapping Dictionaries:** Neural networks only understand numbers. We create two crucial dictionaries:
    * `char2int`: Maps each unique character to an integer ID. This is used to encode our training data.
    * `int2char`: Maps the integer IDs back to characters. We will use this later to decode the model's predictions into readable text.
* **Encoding the Data:** Finally, we loop through the entire raw text and convert it into `encoded_text`, a single, massive NumPy array of integers ready to be sliced into training sequences.

In [52]:
response = requests.get(URL)
text = response.text
text = text[:100000]

chars = tuple(set(text))
int2char = dict(enumerate(chars))
char2int = {ch: i for i, ch in int2char.items()}

encoded_text = np.array([char2int[ch] for ch in text])

VOCAB_SIZE = len(chars)

## Structuring the Data: The Batch Generator

To train our LSTM efficiently, we cannot feed it the entire text at once, nor can we feed it one character at a time. We must package our data into continuous, parallel streams called batches.

The `get_batches` function is a Python generator designed to yield these batches memory-efficiently:

* **Trimming and Reshaping:** First, it calculates the maximum number of full batches we can extract and discards any leftover characters at the end of the text. The data is then reshaped into a matrix where each row represents a separate stream of text running in parallel.

* **The Input ($X$) and Target ($Y$):** For language modeling, the network's goal is always to predict the very next character. 
  * The input array $X$ contains a chunk of text of length `SEQ_LENGTH`.
  * The target array $Y$ is a copy of $X$, but shifted exactly one character into the future. 

* **Example:** If our text is "HELLO WORLD" and our sequence length is 4:
  * $X$ might be `[H, E, L, L]`
  * $Y$ will be `[E, L, L, O]`

By shifting the target array, we create a supervised learning environment: for every character the LSTM sees in $X$, it is penalized if it doesn't correctly predict the corresponding character in $Y$.

In [53]:
def get_batches(arr, batch_size, seq_length):
    batch_size_total = batch_size * seq_length
    n_batches = len(arr) // batch_size_total
    arr = arr[:n_batches * batch_size_total]

    arr = arr.reshape((batch_size, -1))

    for n in range(0, arr.shape[1], seq_length):
        x = arr[:, n:n+seq_length]
        y = np.zeros_like(x)

        try:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, n+seq_length]
        except IndexError:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, 0]

        yield x, y


batches = get_batches(encoded_text, BATCH_SIZE, SEQ_LENGTH)
x_sample, y_sample = next(batches)

## Building the Character-Level Architecture

We define our neural network by creating a custom class that inherits from PyTorch's `nn.Module`. While the core contains an LSTM, the surrounding layers are specifically tailored for processing discrete text (classification) rather than continuous numbers (regression).

### The Layers (`__init__`)

* **`nn.Embedding`:** This is the most crucial addition for Natural Language Processing. Instead of feeding raw integer IDs directly into the LSTM, the embedding layer acts as a lookup table. It maps each character's integer ID to a dense, trainable vector of size `EMBEDDING_DIM`. This allows the network to learn spatial relationships between characters.
* **`nn.LSTM`:** The recurrent core that processes the sequence of embedded vectors and updates its memory states. 
* **`nn.Linear`:** The final fully connected layer. It takes the LSTM's hidden state and expands it to match our `VOCAB_SIZE`. The output of this layer represents the raw scores (logits) for every possible character in our vocabulary.

### The Forward Pass (`forward`)

The `forward` method defines the journey of our input data (`x`) and the previous memory state (`hidden`):

1. **Embedding:** The raw integer sequence is converted into rich vector representations.
2. **Recurrence:** The embedded sequence and the previous hidden states are passed through the LSTM. Crucially, we return the new `hidden` state so it can be passed to the next batch, allowing the model to maintain a memory across long texts.
3. **Reshaping:** Before passing the LSTM output to the Linear layer, we flatten the batch and sequence dimensions using `reshape(-1, self.hidden_size)`. This is required because our final goal is classification; our loss function will expect a 2D tensor where every single row is a distinct prediction, and every column corresponds to a character in the vocabulary.
4. **Scoring:** The reshaped tensor is passed through the `fc` layer to produce the final predictions.

### Initializing Memory (`init_hidden`)

Unlike feedforward networks, LSTMs require an initial state to start calculating. The `init_hidden` method creates two tensors of zeros (the hidden state $h_0$ and the cell state $c_0$) to act as a blank memory slate at the beginning of an epoch or during text generation.

In [54]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        embedded = self.embedding(x)
        lstm_out, hidden = self.lstm(embedded, hidden)
        lstm_out = lstm_out.reshape(-1, self.hidden_size)
        output = self.fc(lstm_out)

        return output, hidden

    def init_hidden(self, batch_size, device):
        return (torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device),
                torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device))

model = CharLSTM(vocab_size=VOCAB_SIZE, embedding_dim=EMBEDDING_DIM, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS).to(DEVICE)
print(model)

CharLSTM(
  (embedding): Embedding(61, 128)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True)
  (fc): Linear(in_features=256, out_features=61, bias=True)
)


## Training the Language Model

Training an LSTM on continuous text sequences requires a few special considerations compared to standard feedforward networks. Here is a breakdown of the key components in our training loop:

### 1. Classification vs. Regression
Unlike predicting a continuous sine wave, predicting a character is a multi-class classification problem. Therefore, we use **`nn.CrossEntropyLoss()`**. The model outputs raw scores for every character in our vocabulary, and this loss function penalizes the model if the highest score doesn't belong to the correct target character.

### 2. State Detachment (Truncated Backpropagation)
Notice this crucial line inside the batch loop: 
`h = tuple([each.data for each in h])`

Because we want the model to remember context across batches, we pass the hidden state `h` from one batch to the next. However, PyTorch tracks every operation to calculate gradients. If we didn't detach the hidden state, PyTorch would try to backpropagate through the entire text we've seen so far, quickly running out of memory. This line creates a fresh copy of the state's *values* while forgetting its computational history.

### 3. Target Reshaping
When calculating the loss: `criterion(output, targets.view(-1))`
Our model's output was flattened in the `forward` pass to a 2D tensor of shape `[batch_size * seq_length, vocab_size]`. To match this, we must also flatten our targets into a 1D tensor of shape `[batch_size * seq_length]` using `.view(-1)`.

### 4. Gradient Clipping
LSTMs are prone to the **Exploding Gradient Problem**, where weight updates become exponentially large and break the model. The line `nn.utils.clip_grad_norm_(model.parameters(), CLIP)` acts as a safety valve, capping the maximum size of the gradients before the optimizer updates the weights.

In [55]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

model.train()
for epoch in range(NUM_EPOCHS):
    h = model.init_hidden(BATCH_SIZE, DEVICE)

    for x, y in get_batches(encoded_text, BATCH_SIZE, SEQ_LENGTH):
        inputs = torch.from_numpy(x).long().to(DEVICE)
        targets = torch.from_numpy(y).long().to(DEVICE)

        h = tuple([each.data for each in h])

        optimizer.zero_grad()
        output, h = model(inputs, h)
        loss = criterion(output, targets.view(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        optimizer.step()

    print(f"Epoch: {epoch+1}/{NUM_EPOCHS}, Loss: {loss.item():.4f}")

Epoch: 1/20, Loss: 3.2839
Epoch: 2/20, Loss: 3.0566
Epoch: 3/20, Loss: 2.7343
Epoch: 4/20, Loss: 2.4885
Epoch: 5/20, Loss: 2.3481
Epoch: 6/20, Loss: 2.2456
Epoch: 7/20, Loss: 2.1603
Epoch: 8/20, Loss: 2.0861
Epoch: 9/20, Loss: 2.0181
Epoch: 10/20, Loss: 1.9626
Epoch: 11/20, Loss: 1.9102
Epoch: 12/20, Loss: 1.8665
Epoch: 13/20, Loss: 1.8266
Epoch: 14/20, Loss: 1.7936
Epoch: 15/20, Loss: 1.7652
Epoch: 16/20, Loss: 1.7298
Epoch: 17/20, Loss: 1.6982
Epoch: 18/20, Loss: 1.6753
Epoch: 19/20, Loss: 1.6497
Epoch: 20/20, Loss: 1.6262


## Inference: Generating New Text

Once the model has learned the patterns of the language, we can use it to generate completely new text. This process is called **Inference**.

### The `predict` Function: One Step at a Time

Generating text happens exactly one character at a time. The `predict` function takes a single character, processes it through the model, and returns the next character. 

Instead of just picking the character with the absolute highest score (which leads to repetitive, boring loops), we treat the output as a probability distribution and randomly sample from it. Two key techniques control this sampling:

1. **Temperature:** Before applying the Softmax function to convert the raw scores (logits) into probabilities, we divide the logits by a `temperature` value.
   * **High Temperature (e.g., 1.2):** Shrinks the difference between scores. The probabilities become more uniform, meaning the model is more likely to choose "unexpected" characters. This increases creativity but also increases spelling mistakes and chaos.
   * **Low Temperature (e.g., 0.5):** Exaggerates the difference between scores. The model becomes highly confident and conservative, choosing only the most likely characters. The text will be grammatically correct but potentially repetitive.

2. **Top-K Filtering:** Even with temperature, we don't want the model to accidentally choose a character that makes zero sense. If `top_k=5`, we force the probability of all characters outside the top 5 highest scores to be 0. We then re-normalize the remaining 5 probabilities. This ensures the model is "creative" only within a safe boundary of sensible choices.

### The `generate` Function: Autoregression

The `generate` function builds the full text using an autoregressive loop:
1. **Priming:** It starts with a `prime` string (e.g., "The "). It feeds these characters into the model one by one to build up a meaningful hidden state (memory) without recording the predictions.
2. **Generating:** It takes the final prediction from the prime, adds it to the output string, and then feeds *that newly predicted character* right back into the model to get the next one. It repeats this loop `size` times.

In [56]:
def predict(model, char, h=None, top_k=None, temperature=1.0):
    model.eval()

    x = np.array([[char2int[char]]])
    input = torch.from_numpy(x).long().to(DEVICE)

    if h is not None:
        h = tuple([each.data for each in h])

    out, h = model(input, h)
    logits = out[-1] / temperature

    p = F.softmax(logits, dim=0).detach().cpu().numpy()

    if top_k is not None:
        p[np.argsort(p)[:-top_k]] = 0
        p = p / p.sum()

    char_index = np.random.choice(len(chars), p=p)

    return int2char[char_index], h

def generate(model, size=500, prime='The ', top_k=5, temperature=1.0):
    model.eval()

    chars_generated = [ch for ch in prime]
    h = model.init_hidden(1, DEVICE)

    for ch in prime:
        char, h = predict(model, ch, h, top_k=top_k, temperature=temperature)

    chars_generated.append(char)

    for _ in range(size - 1):
        char, h = predict(model, chars_generated[-1], h, top_k=top_k, temperature=temperature)
        chars_generated.append(char)

    return ''.join(chars_generated)

## Analyzing the Output: The Impact of Temperature

By running our `generate` function across a spectrum of temperature values, we can clearly observe how manipulating the Softmax probabilities fundamentally changes the character of the generated text. 

Here is a breakdown of the model's behavior at each stage, based on the generated outputs:

### 1. Extremely Conservative (Temperature = 0.25)
* **Observation:** The model falls into a massive repetitive loop, endlessly generating phrases like *"the people"* and *"the pears"*. 
* **Why this happens:** At a low temperature, the model is mathematically terrified of taking risks. It almost exclusively selects the single most probable next character. Because "the" and "people" are incredibly common in political Shakespearean texts (like *Coriolanus*), the model finds a safe path and refuses to leave it. The grammar is technically safe, but the text is useless.

### 2. Cautious but Expanding (Temperature = 0.50)
* **Observation:** The repetition decreases significantly. We start seeing actual play formatting (like the character name `CORIOLANUS:` appearing). The vocabulary expands ("shollow", "stand", "words"), but the sentence structures still feel slightly rigid and looping.
* **Why this happens:** The model is allowed slightly more variance, breaking the "the people" loop, but it still heavily favors high-probability word structures.

### 3. The "Sweet Spot" (Temperature = 0.75)
* **Observation:** This output looks the most like actual Shakespeare. We see a realistic mix of character tags (`MARCIUS:`, `BRUTUS:`), varied sentence lengths, and natural-looking punctuation. While there are spelling mistakes and pseudo-words ("cranstaly", "presceld"), the rhythm and flow feel structurally sound.
* **Why this happens:** This is generally the optimal zone for character-level models. The temperature is high enough to allow the model to traverse different vocabulary paths and take creative risks, but low enough that the fundamental rules of English spelling learned during training are still respected.

### 4. Overly Creative / Degrading (Temperature = 1.0)
* **Observation:** The structure begins to collapse. While the model correctly formats capitalization and spacing, the actual words are heavily misspelled ("themsers", "oun mane", "widh"). It starts to lose track of long-term dependencies.
* **Why this happens:** The model is frequently picking the 2nd, 3rd, or 4th most likely characters instead of the optimal ones. It is trying to be too "creative" with its spelling, resulting in phonetic but incorrect English.

### 5. Pure Chaos (Temperature = 1.25)
* **Observation:** The text devolves into nearly complete gibberish ("havigher", "strivuler", "cherthice"). The punctuation becomes erratic, and coherent words are rare.
* **Why this happens:** The probability distribution has been flattened so much that the model is actively forced to pick characters that have very little logical business being next to each other. It successfully generates things that *look* like English words at a quick glance, but hold zero structural integrity.

### Summary Takeaway
This experiment perfectly highlights the core challenge of Generative AI: finding the exact balance between **strict adherence to learned rules (low temperature)** and **the ability to generate novel, unseen sequences (high temperature)**. 

In [57]:
temperatures = [0.25, 0.5, 0.75, 1.0, 1.25]

for temp in temperatures:
    print(f"\nTemperature: {temp}")
    generated_text = generate(model, size=400, prime='KING:', top_k=5, temperature=temp)
    print(generated_text)


Temperature: 0.25
KING:
The can the comparing the martion,
And the common the people the people and the present the have so propered them
The people and the people and the words and the strends we the people
The common the people the people, and the common the common the people a man as the consul
The common the people the people and the pear the comparise the people the see man the pear
The content the common the peopl

Temperature: 0.5
KING:
I would shall so, and the people,
Where with the people the spate the worst him a sollone.

MENENIUS:
The commin of the people, and the here a prods, and the pear
The people to the people.

COMINIUS:
He should be sender for him.

CORIOLANUS:
When you he shall for which honours
The people to have here as a contricaning.

MENENIUS:
What he seed the heart the worst the proves to consul,
And the prom

Temperature: 0.75
KING:
He share see and the consul
Mances worthy seed. Where has me.

BRUTUS:
He show all the partion the pores a pear the cremple, a